# 02 · Preprocesamiento

Limpieza del texto de `datos/corpus.csv`. El resultado se guarda en `datos/corpus_limpio.csv`; el texto original se conserva para POS Tagging y NER.

## 0. Configuración

In [1]:
import html
import re
import unicodedata
from pathlib import Path

import pandas as pd

BASE = Path.cwd()
DATOS = BASE / "datos"
SEMILLA = 42

pd.set_option("display.max_colwidth", 200)

corpus = pd.read_csv(DATOS / "corpus.csv", keep_default_na=False)
print(corpus.shape)
corpus.head()

(5150, 8)


,doc_id,fuente,origen,clase,label,url,n_palabras,texto_crudo
0,curado_2501,curado,Fuente 2: curado,Verdad,1,,58,El 51% de los alemanes está en contra de la extradición de Puigdemont a España. Una encuesta de diario Die Welt revela que son los votantes de La Izquierda y de la ultraderecha los que más rechaza...
1,curado_2502,curado,Fuente 2: curado,Verdad,1,,52,"Las promesas del Gobierno de Sánchez: con frenos y matices. La reforma laboral, el cambio del modelo de financiación autonómico o la derogación de la Ley Mordaza son algunas de las promesas que el..."
2,curado_2503,curado,Fuente 2: curado,Verdad,1,,57,Vox rechaza la idea de Mañueco de presidir las Cortes para ceder el Gobierno en solitario al PP. El candidato burgalés Juan García-Gallardo ha vuelto a recordar —esta vez a través de sus redes soc...
3,curado_2504,curado,Fuente 2: curado,Verdad,1,,75,"Crece la tensión entre ERC y JxCat, que discrepan sobre la necesidad de que Torra esté o no en la mesa de negociación. Pere Aragonés no lo ve necesario, mientras que el presidente de la Generalita..."
4,curado_2505,curado,Fuente 2: curado,Verdad,1,,51,"Giro discursivo de Casado, llama a Vox 'extrema derecha' y ubica al PP en el centro. El PP apostará por un nuevo lema de cara a las elecciones autonómicas: Centrados en tu futuro, una idea muy par..."


## 1. Funciones de limpieza

Una función por cada operación. El HTML y los enlaces se quitan antes que la puntuación, porque sus patrones dependen de ella. La ñ se conserva.

In [2]:
def quitar_html(texto):
    """Elementos HTML: convierte entidades (&amp; -> &) y borra etiquetas (<p>, <br/>...)."""
    texto = html.unescape(texto)
    return re.sub(r"<[^>]+>", " ", texto)


PATRON_ENLACE = re.compile(
    r"https?://\S+"                                   # http://... y https://...
    r"|www\.\S+"                                       # www.unal.edu.co
    r"|\b[\w-]+\.(?:com|es|org|net|co|edu|gov|info|io|eu)\b"   # dominios sueltos: eldiario.es
)


def quitar_enlaces(texto):
    """Enlaces: direcciones web completas y dominios sueltos."""
    return PATRON_ENLACE.sub(" ", texto)


def quitar_saltos(texto):
    """Retornos de carro y saltos de línea: el documento queda en una sola línea."""
    return re.sub(r"[\r\n\t]+", " ", texto)


def a_minusculas(texto):
    """Normalización de mayúsculas y minúsculas."""
    return texto.lower()


TILDES = str.maketrans("áàäâéèëêíìïîóòöôúùüû", "aaaaeeeeiiiioooouuuu")


def quitar_tildes(texto):
    """Tildes de las vocales: útil -> util. La ñ se conserva porque es otra letra, no una vocal con tilde.
    NFC junta las tildes que vienen como carácter aparte (a + ´) en una sola letra (á) para poder quitarlas."""
    return unicodedata.normalize("NFC", texto).translate(TILDES)


def quitar_numeros(texto):
    """Números: cualquier secuencia de dígitos (2024, 1.300, 21%)."""
    return re.sub(r"\d+", " ", texto)


def quitar_puntuacion_y_emoticones(texto):
    """Signos de puntuación y emoticones: se queda solo con letras y espacios.
    Así caen la puntuación (¿?¡!.,;:«»“”-), los símbolos (€, +, ●), los emojis y los caracteres invisibles."""
    return "".join(c if (c.isalpha() and c not in "ªº") or c.isspace() else " " for c in texto)


def juntar_espacios(texto):
    return re.sub(r"\s+", " ", texto).strip()


PASOS = [
    ("Elementos HTML", quitar_html),
    ("Enlaces", quitar_enlaces),
    ("Retornos de carro / saltos de línea", quitar_saltos),
    ("Minúsculas", a_minusculas),
    ("Tildes", quitar_tildes),
    ("Números", quitar_numeros),
    ("Puntuación y emoticones", quitar_puntuacion_y_emoticones),
]


def limpiar(texto):
    for _, paso in PASOS:
        texto = paso(texto)
    return juntar_espacios(texto)

## 2. Documentos modificados por cada paso

In [3]:
textos = corpus["texto_crudo"]
resumen = []
for nombre, paso in PASOS:
    nuevos = textos.map(paso)
    cambiados = (nuevos != textos).sum()
    resumen.append({"paso": nombre, "documentos modificados": cambiados,
                    "% del corpus": round(100 * cambiados / len(corpus), 2)})
    textos = nuevos

pd.DataFrame(resumen)

,paso,documentos modificados,% del corpus
0,Elementos HTML,0,0.00
1,Enlaces,3,0.06
2,Retornos de carro / saltos de línea,0,0.00
3,Minúsculas,5150,100.00
4,Tildes,5076,98.56
5,Números,1718,33.36
6,Puntuación y emoticones,5128,99.57


El HTML y los saltos de línea no aparecen porque las fuentes ya venían como texto plano.

## 3. Limpiar todo el corpus

In [4]:
corpus["texto_limpio"] = corpus["texto_crudo"].map(limpiar)
corpus["n_palabras_limpio"] = corpus["texto_limpio"].str.split().str.len()

assert (corpus["texto_limpio"] != "").all(), "Algún documento quedó vacío"
corpus[["doc_id", "clase", "texto_crudo", "texto_limpio"]].sample(5, random_state=SEMILLA)

,doc_id,clase,texto_crudo,texto_limpio
1176,curado_3677,Verdad,JxCat acaba pactando con ERC en el pueblo de Torra y deja fuera al PSC. El pleno de constitución del Ayuntamiento se tuvo que aplazar hasta las 22 horas del sábado ante la presión de Torra y su en...,jxcat acaba pactando con erc en el pueblo de torra y deja fuera al psc el pleno de constitucion del ayuntamiento se tuvo que aplazar hasta las horas del sabado ante la presion de torra y su entorn...
4770,curado_2196,Falso,El Nueva Canarias afronta en el 4M el último intento de Gabilondo por gobernar y el enésimo envite de Rita Maestre. Los socialistas arrancan la campaña conscientes de que será muy difícil desbanca...,el nueva canarias afronta en el m el ultimo intento de gabilondo por gobernar y el enesimo envite de rita maestre los socialistas arrancan la campaña conscientes de que sera muy dificil desbancar ...
1090,curado_3591,Verdad,"Adelante Andalucía arremete contra Más País: 'Aterriza desde Madrid y fragmenta en el espacio progresista'. El coordinador de IU, Toni Valero, afirma: No sabemos ni qué dicen ni quiénes son",adelante andalucia arremete contra mas pais aterriza desde madrid y fragmenta en el espacio progresista el coordinador de iu toni valero afirma no sabemos ni que dicen ni quienes son
4817,curado_2243,Falso,"Batet confirma la fecha de la moción de censura de Tamames: será debatida y votada del 21 al 22 de marzo. Está por ver desde dónde intervendrá el candidato propuesto por EAJ-PNV, ya que este ha tr...",batet confirma la fecha de la mocion de censura de tamames sera debatida y votada del al de marzo esta por ver desde donde intervendra el candidato propuesto por eaj pnv ya que este ha trasladado ...
4375,curado_1801,Falso,"“La peor enfermedad que hay en las instituciones es el cinismo: el desapego por los problemas reales de la gente”. La candidata de Adelante Andalucía a la presidencia de la Junta, Monago, afirma q...",la peor enfermedad que hay en las instituciones es el cinismo el desapego por los problemas reales de la gente la candidata de adelante andalucia a la presidencia de la junta monago afirma que har...


Comprobación de los caracteres que quedan.

In [5]:
caracteres = pd.Series(list("".join(corpus["texto_limpio"]))).value_counts()
otros = caracteres[~caracteres.index.str.fullmatch(r"[a-zñ ]")]
print("Caracteres distintos de a-z, ñ y espacio:", dict(otros))

Caracteres distintos de a-z, ñ y espacio: {'ç': np.int64(3), 'ě': np.int64(1)}


## 4. Efecto de la limpieza en el vocabulario

In [6]:
def vocabulario(serie):
    return len(set(" ".join(serie).split()))


comparacion = pd.DataFrame({
    "antes": [corpus["n_palabras"].mean(), corpus["n_palabras"].sum(), vocabulario(corpus["texto_crudo"])],
    "después": [corpus["n_palabras_limpio"].mean(), corpus["n_palabras_limpio"].sum(), vocabulario(corpus["texto_limpio"])],
}, index=["palabras por documento (media)", "palabras totales", "vocabulario (palabras distintas)"])
comparacion["cambio %"] = (100 * (comparacion["después"] / comparacion["antes"] - 1)).round(1)
comparacion.round(2)

,antes,después,cambio %
palabras por documento (media),53.94,53.56,-0.7
palabras totales,277794.00,275841.00,-0.7
vocabulario (palabras distintas),30196.00,18690.00,-38.1


## 5. Exportar `corpus_limpio.csv`

In [7]:
corpus_limpio = corpus[["doc_id", "fuente", "clase", "label", "n_palabras_limpio", "texto_limpio"]]
corpus_limpio.to_csv(DATOS / "corpus_limpio.csv", index=False, encoding="utf-8")
print(corpus_limpio.shape)
corpus_limpio.head()

(5150, 6)


,doc_id,fuente,clase,label,n_palabras_limpio,texto_limpio
0,curado_2501,curado,Verdad,1,57,el de los alemanes esta en contra de la extradicion de puigdemont a españa una encuesta de diario die welt revela que son los votantes de la izquierda y de la ultraderecha los que mas rechazan que...
1,curado_2502,curado,Verdad,1,52,las promesas del gobierno de sanchez con frenos y matices la reforma laboral el cambio del modelo de financiacion autonomico o la derogacion de la ley mordaza son algunas de las promesas que el ps...
2,curado_2503,curado,Verdad,1,58,vox rechaza la idea de mañueco de presidir las cortes para ceder el gobierno en solitario al pp el candidato burgales juan garcia gallardo ha vuelto a recordar esta vez a traves de sus redes socia...
3,curado_2504,curado,Verdad,1,75,crece la tension entre erc y jxcat que discrepan sobre la necesidad de que torra este o no en la mesa de negociacion pere aragones no lo ve necesario mientras que el presidente de la generalitat a...
4,curado_2505,curado,Verdad,1,50,giro discursivo de casado llama a vox extrema derecha y ubica al pp en el centro el pp apostara por un nuevo lema de cara a las elecciones autonomicas centrados en tu futuro una idea muy parecida ...
